In [ ]:
#@title 按這裡開始（先按 ▶）
print("✅ W16 出發！本週目標：用程式批次整理資料，並產生 data_list.csv")
print("這一本是 Colab 備援版；教室電腦裝得起 Python 的人請用 scripts/batch_prepare.py")
print("本週要自己補三個空：新檔名、新檔的完整路徑、把圖縮成同一個尺寸")

# W16　資料蒐集與批次整理（電腦教室版 · Colab 備援）

**兩條路，程式完全一樣，只差 `ROOT` 那一行**

| 步驟 | 教室電腦本機 Python | 這一本（Colab 備援） |
|---|---|---|
| 環境 | 開始功能表 → Anaconda Prompt | Chrome 開 Colab，什麼都不用裝 |
| 裝套件 | `pip install pillow` | `!pip install -q pillow`（多半已內建） |
| 資料放哪 | 本機 `D:\ai_project\raw` | 雲端硬碟 `MyDrive/AI115/raw` |
| 怎麼跑 | 存成 `batch.py`，`python batch.py` | 貼進儲存格按 ▶ |

**教室電腦多半沒有安裝權限，五分鐘內裝不起來就直接走這一本，不影響分數。**

> ⚠️ **跑之前一定要先複製一份原始資料。**
> `os.rename()` 與 `im.save()` 都是直接覆蓋原檔，改壞了救不回來。

**資料夾要長這樣**（一個類別一個子資料夾，名字一律用英文與底線）：

```
raw/
  paper/    IMG_0012.jpg ...
  bottle/   ...
  can/      ...
```

**iPhone 拍的 HEIC 檔 Pillow 打不開**，會被當成壞檔；
請先在相機設定改成「相容性最佳」，或上傳前轉成 JPG。

### 設定：掛載雲端硬碟並指定 `ROOT`

**這一格要做什麼**：裝 Pillow、掛載雲端硬碟、把 `ROOT` 指到你們的 `raw` 資料夾。

**寫對了會看到什麼**：印出 `ROOT = ...`，以及一個類別資料夾的清單，
例如 `['paper', 'bottle', 'can']`。

**印出空的 list 或說找不到路徑**：回雲端硬碟網頁版確認資料夾真的叫這個名字，
路徑大小寫要一模一樣。

In [ ]:
#@title 設定 ROOT（投影片的「兩條路」那一頁）
!pip install -q pillow                    # 多半已內建，跑一下確認
from google.colab import drive            # ←投影片未含，執行所需
drive.mount("/content/drive")             # ←投影片未含，執行所需
import os
ROOT = "/content/drive/MyDrive/AI115/raw"  # ← 走本機就換成 r"D:\ai_project\raw"
print("ROOT =", ROOT)
print("類別資料夾：", sorted(os.listdir(ROOT)))

### 第 1 段：批次改檔名

改檔名的目的：讓每個檔案一看就知道是哪一類、第幾張，程式才好讀。
目標長這樣 → `paper_0001.jpg`（類別名＋底線＋四位數編號＋原本的副檔名，全部小寫）。

**這一格要做什麼**：補兩行。
- 第一行：組出 `cls_0001.jpg` 這種名字。提示：`{i:04d}` 是「補零到四位數」，1 會變成 0001。
- 第二行：`os.rename` 的第二個參數要**完整路徑**，只給檔名檔案會被搬到工作目錄去。

**寫對了會看到什麼**：每一類印一行「paper 100 個檔案改名完成」，
回雲端硬碟看檔名全部變整齊了。

**`enumerate(files, 1)` 的 1** 是起始編號，所以第一張是 0001 不是 0000。

In [ ]:
#@title 第 1 段：批次改檔名（跑之前先複製一份原始資料！）
import os, glob
for cls in os.listdir(ROOT):
    d = os.path.join(ROOT, cls)
    files = sorted(glob.glob(os.path.join(d, "*")))
    for i, old in enumerate(files, 1):
        ext = os.path.splitext(old)[1].lower()
        new = ____          # ← 自己寫：cls_0001.jpg 這種名字
        os.rename(old, ____)   # ← 自己寫：新檔的完整路徑
    print(cls, len(files), "個檔案改名完成")

### 第 2 段：統一尺寸並找出壞檔

**這一格要做什麼**：補一行——把圖縮成 `SIZE` 那麼大。
提示：Pillow 的方法名稱就是「重新調整大小」。

**寫對了會看到什麼**：印出「壞檔 N 個」，後面列出打不開的檔案路徑。
**這份清單要自己手動刪掉**，程式不會替你刪。

**為什麼要 `convert("RGB")`**：PNG 有透明色版、灰階圖只有一個通道，
統一轉成 RGB 之後，第 17 週訓練才不會因為通道數不同而報錯。

**`Image 說檔案壞掉`**＝下載到一半中斷，那就是真的壞檔，刪掉並回頭補拍。

In [ ]:
#@title 第 2 段：統一尺寸並找出壞檔
from PIL import Image
SIZE = (224, 224)
bad = []
for f in glob.glob(os.path.join(ROOT, "*", "*")):
    try:
        im = Image.open(f).convert("RGB")
        im = ____              # ← 自己寫：縮成 SIZE 大小
        im.save(f, quality=90)
    except Exception:
        bad.append(f)
print("壞檔", len(bad), "個")
for f in bad:
    print("  ", f)

### 第 3 段：產生 `data_list.csv`

這份清單第 17 週訓練時要用，也是本週的繳交項目。沒有空格，直接執行。

**寫對了會看到什麼**：印出「共 NNN 筆，已寫出 data_list.csv」，
`raw` 資料夾裡多一個 `data_list.csv`（三欄：檔名、類別、KB）。

**`encoding="utf-8-sig"` 不能省**：少了它，
用 Excel 打開中文欄名會變成一堆亂碼。

In [ ]:
#@title 第 3 段：產生 data_list.csv
import csv
rows = []
for cls in sorted(os.listdir(ROOT)):
    for f in sorted(glob.glob(os.path.join(ROOT, cls, "*"))):
        rows.append([os.path.basename(f), cls,
                     os.path.getsize(f) // 1024])
out = os.path.join(ROOT, "data_list.csv")
with open(out, "w", newline="", encoding="utf-8-sig") as fp:
    w = csv.writer(fp)
    w.writerow(["檔名", "類別", "KB"])
    w.writerows(rows)
print("共", len(rows), "筆，已寫出 data_list.csv")

### 第 4 段（進階）：用雜湊找出重複

內容一模一樣的檔案雜湊值會相同，**改了檔名也躲不掉**。沒有空格，直接執行。

**寫對了會看到什麼**：印出「重複的檔案 N 組」，並列出前十組的檔名對照。

**為什麼要找重複**：同一張圖同時出現在訓練與測試裡，
測試分數會虛高——模型只是「背過答案」而已。找出來一組只留一張。

In [ ]:
#@title 第 4 段（進階）：用雜湊找重複
import hashlib
seen, dup = {}, []
for f in glob.glob(os.path.join(ROOT, "*", "*")):
    h = hashlib.md5(open(f, "rb").read()).hexdigest()
    if h in seen:
        dup.append((f, seen[h]))
    else:
        seen[h] = f
print("重複的檔案", len(dup), "組")
for a, b in dup[:10]:
    print(os.path.basename(a), "==", os.path.basename(b))

### 收工：延伸挑戰與繳交

- **A**（每個人都要做完）：在 `data_list.csv` 多加一欄「寬×高」，
  確認統一尺寸真的成功了（提示：`Image.open(f).size`）。
- **B**：自己寫一個 `report()`，讀 `data_list.csv` 印出各類筆數、
  最多最少差幾倍、壞檔幾個。第 15 週的 `check_bias()` 可以直接拿來接。
- **C**：說出你們這份資料最可能的偏誤在哪——角度、光線、地點還是人，
  並寫下打算怎麼補。這一題第 17 週會拿出來對照。

**標註四件事**：先寫下規則再開始標；邊界案例先討論；
一種類別只由一個人標；標完互相抽查 20 筆。

**常見狀況**：pip 說權限不足＝直接走這一本；
改完檔名全跑掉＝第二行只填了檔名，用複本重來；
CSV 開起來是亂碼＝寫檔時忘了 `utf-8-sig`。

In [ ]:
#@title 收工檢查（直接按 ▶）
print("本週要交：整理好的資料夾與 data_list.csv、各類筆數統計與最多最少差幾倍")
print("以及 batch.py 或這一本整理用的 .ipynb")
print("檔名：AI導論_W16_學號_姓名，資料下課前放進小組共用資料夾")
print("提醒：整理完立刻複製回雲端硬碟，教室電腦重開機會還原")

---

<details>
<summary>參考解（三個空格都自己試過再打開）</summary>

```python
# 第 1 段
        new = f"{cls}_{i:04d}{ext}"
        os.rename(old, os.path.join(d, new))

# 第 2 段
        im = im.resize(SIZE)
```

為什麼是這樣寫：

- `{i:04d}` 是「整數補零到四位數」，1 → `0001`。
  補零之後檔名排序才會照數字順序，不然 `10` 會排在 `2` 前面。
- `os.rename()` 的第二個參數要**完整路徑**。只填 `new` 的話，
  檔案會被搬到程式當下的工作目錄，整批資料就散掉了。
- `im.resize(SIZE)` 會回傳一張新圖，**要接回 `im`**；
  寫成 `im.resize(SIZE)` 而不接回去，存出來的還是原尺寸。

</details>